In [1]:
%run func_defs.ipynb

%run NN_defs.ipynb

import time, os, random
seed = 42

start_new_nn = 1
ns = 300 ;
nz = 640; N_samples= 8100
Lz=10 ; tf = 20; 
k1=0.5; k2= 1
lr = .01
wd = .0001
epochs = 10000000
betas = (0.995, 0.999)
tol = 5e-08
#nodes = 32
out_file= f'./model/2layer_tol{tol}_sobol.pt'
dat_file = f'./data/training_data_4d.pt'
#dat_file = f'./data/2layer_k1{k1}_k2{k2}_ns{ns}.pt'

d_in= 4
#2layer_k11.0_k22.0_nz640.pt

In [6]:
# set a dictionary of parameters
params_def = {'d_in':4, 'd_out':1, 'layers':4, 'nodes':64,
    'tol':0.5e-6, 'epochs':1000000, 'seg_size':1000, 'epoch_0':0,
    'lr':0.005, 'eps':1e-10, 'betas':(0.995, 0.999), 'wd':1e-4,
    'L':10, 'N':100000, 'N_test':100000,
    'start_new_nn':1, 'dat_file':f'./data/tm_in.pt',
    'out_file':f'./data/tm_out.pt'} 
for name in params_def: # using existing unless not existent
    if name not in globals():
        globals()[name] = params_def[name]

sec_main = """
# Load the training data


device = torch.device('cuda')
# ---- Load the dataset ----
d = torch.load('training_data_4d.pt', map_location=device)
X_train = d['X'].to(device)      # columns: k1, z_d, s, tau
Y_train = d['Y'].to(device)      # W = U / U0
print(d['cols'], '->', d['target'])
print('X:', tuple(X_train.shape), ' Y:', tuple(Y_train.shape), '|', X_train.device)


Y_train = Y_train.reshape(len(Y_train),1)


xtrain = X_train.to(torch.float64)
ytrain = Y_train.to(torch.float64)
print(ytrain.min().item(), ytrain.max().item(), ytrain.mean().item(), ytrain.std().item())
print('finite:', torch.isfinite(ytrain).all().item())
# how heavy is the tail?
if start_new_nn == 1:
    model, optimizer = build_nn() # build a new model
    model = model.to(device)
    del start_new_nn
else:
    model, optimizer = rebuild_nn() # rebuild the model from saved data
    model = model.to(device)
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(device)
print(f'neural network model is on {next(model.parameters()).device}')
print_params() # print parameters
#
trained_model = train_nn() # train the model
for key in trained_model: # unpack trained_model
    globals()[key] = trained_model[key]
# test_result = test_and_plotting() # test the model and plot results
# for key in test_result: # unpack test_result
   # globals()[key] = test_result[key]
#
print_params()
[print(msg) for msg in msg_train]
# [print(msg) for msg in msg_test
save_model_vars() # save the model and variables in a dictionary
"""


In [7]:
# run the code unless func_def_only ==1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

exec(sec_main)


<string>:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


['k1', 'z_d', 's', 'tau'] -> W = U / U0
X: (20480, 4)  Y: (20480,) | cuda:0
0.6575777942063904 1.0042145873866468 0.9432091119228198 0.07058119433109822
finite: True
neural network model is on cuda:0
Structure of neural network:  Net(
  (layers): ModuleList(
    (0): Linear(in_features=4, out_features=64, bias=True)
    (1-3): 3 x Linear(in_features=64, out_features=64, bias=True)
  )
  (out): Linear(in_features=64, out_features=1, bias=True)
)
Optimizer:   AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.995, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-10
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0.0001
)
parameters: d_in = 4, d_out = 1, layers = 4, nodes = 64
    tol = 5.0000e-08, epochs = 10000000, seg_size = 1000, epoch_0 = 0
    N_train = 100000, N_test = 100000
    output file: ./model/2layer_tol5e-08_sobol.pt

i = 0, loss = 3.1051e-01 loss_curr_min = 3.1051e-01
Elapsed time = 0.00s,  CA time = Sat Aug 29